You can also follow along on Google Colab!

<a target="_blank" href="https://colab.research.google.com/github/MadryLab/context-cite/blob/main/notebooks/quickstart_example.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Quickstart example for `ContextCite`

In this notebook, we'll provide an overview of the ContextCite API by going through a simple example. **If running in Colab, be sure to change your to a GPU runtime!**

Let's start by installing the library:

In [1]:
!pip install context-cite
# !pip install accelerate
# !pip install transformers==4.38.2 --upgrade --force-reinstall
# !pip uninstall -y numpy transformers tokenizers torch

In [2]:
import accelerate
print(accelerate.__version__)

import transformers
print(transformers.__version__)


/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1.6.0
4.51.3


We will use the `ContextCiter` class to attribute models' responses to sources within the context we provide to them.

In [3]:
import context_cite
from transformers import AutoTokenizer

from context_cite import ContextCiter

[nltk_data] Downloading package punkt_tab to /data/healthy-
[nltk_data]     ml/scratch/yuexing/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


For this example, we'll use a TinyLlama chat model.

In [4]:
import pandas as pd

# Load the CSV file
df_op4 = pd.read_csv("medbullets_op4.csv")

# Count duplicates before removing
num_duplicates = df_op4.duplicated(subset=['question']).sum()

# Remove duplicate rows based on the 'question' column, keeping the first occurrence
df_op4 = df_op4.drop_duplicates(subset=['question'], keep='first')

# Display the number of duplicate rows found
print(f"Number of duplicate rows removed: {num_duplicates}")

# Display the first few rows after removing duplicates
# print(df_op4.columns)
# print(df_op4.head())
print(len(df_op4))

Number of duplicate rows removed: 10
298


In [16]:
model_name_or_path = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B" ##TinyLlama/TinyLlama-1.1B-Chat-v1.0

context = """
Attention Is All You Need

Abstract
The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-to-German translation task, improving over the existing best results, including ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task, our model establishes a new single-model state-of-the-art BLEU score of 41.8 after training for 3.5 days on eight GPUs, a small fraction of the training costs of the best models from the literature. We show that the Transformer generalizes well to other tasks by applying it successfully to English constituency parsing both with large and limited training data.
1 Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks in particular, have been firmly established as state of the art approaches in sequence modeling and transduction problems such as language modeling and machine translation [35, 2, 5]. Numerous efforts have since continued to push the boundaries of recurrent language models and encoder-decoder architectures [38, 24, 15].
Recurrent models typically factor computation along the symbol positions of the input and output sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden states ht, as a function of the previous hidden state ht-1 and the input for position t. This inherently sequential nature precludes parallelization within training examples, which becomes critical at longer sequence lengths, as memory constraints limit batching across examples. Recent work has achieved significant improvements in computational efficiency through factorization tricks [21] and conditional computation [32], while also improving model performance in case of the latter. The fundamental constraint of sequential computation, however, remains.
Attention mechanisms have become an integral part of compelling sequence modeling and transduction models in various tasks, allowing modeling of dependencies without regard to their distance in the input or output sequences [2, 19]. In all but a few cases [27], however, such attention mechanisms are used in conjunction with a recurrent network.
In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Transformer allows for significantly more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs.
"""
query = "What type of GPUs did the authors use in this paper?"

In [5]:
import re

def split_question(question_text):
    """Split question text into context and query (last sentence)."""
    sentences = re.split(r'(?<=[.?!])\s+', question_text.strip())
    if len(sentences) <= 1:
        return "", question_text.strip()
    context = " ".join(sentences[:-1])
    query = sentences[-1]
    return context, query

# Loop through each row and construct the input
for idx, row in df_op4.iterrows():
    full_question = row['question']
    opa, opb, opc, opd = row['opa'], row['opb'], row['opc'], row['opd']

    # Split into context and query
    context_text, query_text = split_question(full_question)

    # Combine query with answer options
    query_full = f"{query_text}\nA. {opa}\nB. {opb}\nC. {opc}\nD. {opd}"

    # Optional: show result for one row
    print(f"\n--- Row {idx} ---")
    print("Context:\n", context_text)
    print("Query:\n", query_full)

    # You could now pass `context_text` and `query_full` to ContextCiter, e.g.:
    # response = context_citer.generate_response(context=context_text, query=query_full)

    # Break early to preview just the first result
    # Stop after 5 rows
    if idx >= 4:
        break



--- Row 0 ---
Context:
 A 42-year-old woman is enrolled in a randomized controlled trial to study cardiac function in the setting of several different drugs. She is started on verapamil and instructed to exercise at 50% of her VO2 max while several cardiac parameters are being measured.
Query:
 During this experiment, which of the following represents the relative conduction speed through the heart from fastest to slowest?
A. AV node > ventricles > atria > Purkinje fibers
B. Purkinje fibers > ventricles > atria > AV node
C. Purkinje fibers > atria > ventricles > AV node
D. Purkinje fibers > AV node > ventricles > atria

--- Row 1 ---
Context:
 A 9-year-old girl presents to the emergency department with a fever and a change in her behavior. She presented with similar symptoms 6 weeks ago and was treated for an Escherchia coli infection. She also was treated for a urinary tract infection 10 weeks ago. Her mother says that last night her daughter felt ill, and her condition has been wors

In [7]:
model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM    
model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto"  # ✅ valid here
    )
tokenizer = AutoTokenizer.from_pretrained(model_id)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


In [ ]:
import os
import re
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM    

# Create output folder if it doesn't exist
output_dir = "Attribution Scores"
os.makedirs(output_dir, exist_ok=True)

# Model to use
# model_name_or_path = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

# Loop over the first 10 rows of df_op4
for idx, row in df_op4.head(298).iterrows():
    full_question = row["question"]
    opa, opb, opc, opd = row["opa"], row["opb"], row["opc"], row["opd"]

    # Split context and query
    context_text, query_text = split_question(full_question)

    # Format the full query prompt
    query_full = (
        f"{query_text}\n"
        f"A. {opa}\nB. {opb}\nC. {opc}\nD. {opd}\n\n"
        "Read the question and state your answer. "
        "State your answer, starting with 'Answer:', ending with two line breaks.\n\n"
    )

    try:
#         # Generate attribution results and response
#         cc = ContextCiter.from_pretrained(
#             model_name_or_path,
#             context=context_text,
#             query=query_full,
#             generate_kwargs={"max_new_tokens": 2048, "do_sample": False} #, "device_map": "auto"
#         )
        # Generate attribution results and response
        cc = ContextCiter(
            model,
            tokenizer,
            context=context_text,
            query=query_full,
            generate_kwargs={"max_new_tokens": 2048, "do_sample": False}
        )

        # Extract structured response
        raw_response = cc.response.strip()
        match = re.search(r"Answer:\s*([A-D])", raw_response)
        extracted_answer = match.group(1).strip() if match else None

        # Get attributions
        result = cc.get_attributions(as_dataframe=True)

        if isinstance(result, pd.io.formats.style.Styler):
            result = result.data

        if isinstance(result, pd.DataFrame):
            qa_id = f"MedBullets df_op4 Q{idx + 1}"
            result["row_index"] = idx
            result["QA_ID"] = qa_id
            result["Extracted_Answer"] = extracted_answer  # Optional: attach model-picked answer

            # Save to individual CSV
            filename = f"{qa_id}.csv".replace(" ", "_")
            result.to_csv(os.path.join(output_dir, filename), index=False)

            print(f"✅ Saved attribution and answer for {qa_id}")
        else:
            print(f"⚠️ Row {idx} returned unexpected type: {type(result)}")

    except Exception as e:
        print(f"❌ Error on row {idx}: {e}")

In [17]:
import os
import re
import pandas as pd

# Output folder
output_dir = "Attribution Scores 1.5B Original"
os.makedirs(output_dir, exist_ok=True)

def split_question(question_text):
    """Split question text into context and query (last sentence)."""
    sentences = re.split(r'(?<=[.?!])\s+', question_text.strip())
    if len(sentences) <= 1:
        return "", question_text.strip()
    context = " ".join(sentences[:-1])
    query = sentences[-1]
    return context, query

# Loop through rows
for idx, row in df_op4.head(298).iterrows():
    full_question = row["question"]

    # Split into context and query
    context_text, query_text = split_question(full_question)

    # Split context_text into sentences
    context_sentences = re.split(r'(?<=[.?!])\s+', context_text.strip())

    # Build DataFrame
    sentence_df = pd.DataFrame({
        "Sentence": context_sentences,
        "sentence_ID": list(range(len(context_sentences)))
    })

    # Save to CSV
    qa_id = f"MedBullets df_op4 Q{idx + 1}".replace(" ", "_")
    filename = os.path.join(output_dir, f"{qa_id}.csv")
    sentence_df.to_csv(filename, index=False)

    print(f"✅ Saved sentence order for {qa_id}")


✅ Saved sentence order for MedBullets_df_op4_Q1
✅ Saved sentence order for MedBullets_df_op4_Q2
✅ Saved sentence order for MedBullets_df_op4_Q3
✅ Saved sentence order for MedBullets_df_op4_Q4
✅ Saved sentence order for MedBullets_df_op4_Q5
✅ Saved sentence order for MedBullets_df_op4_Q6
✅ Saved sentence order for MedBullets_df_op4_Q7
✅ Saved sentence order for MedBullets_df_op4_Q8
✅ Saved sentence order for MedBullets_df_op4_Q9
✅ Saved sentence order for MedBullets_df_op4_Q11
✅ Saved sentence order for MedBullets_df_op4_Q12
✅ Saved sentence order for MedBullets_df_op4_Q13
✅ Saved sentence order for MedBullets_df_op4_Q15
✅ Saved sentence order for MedBullets_df_op4_Q16
✅ Saved sentence order for MedBullets_df_op4_Q17
✅ Saved sentence order for MedBullets_df_op4_Q19
✅ Saved sentence order for MedBullets_df_op4_Q20
✅ Saved sentence order for MedBullets_df_op4_Q21
✅ Saved sentence order for MedBullets_df_op4_Q22
✅ Saved sentence order for MedBullets_df_op4_Q23
✅ Saved sentence order for Me

✅ Saved sentence order for MedBullets_df_op4_Q203
✅ Saved sentence order for MedBullets_df_op4_Q204
✅ Saved sentence order for MedBullets_df_op4_Q205
✅ Saved sentence order for MedBullets_df_op4_Q206
✅ Saved sentence order for MedBullets_df_op4_Q207
✅ Saved sentence order for MedBullets_df_op4_Q208
✅ Saved sentence order for MedBullets_df_op4_Q209
✅ Saved sentence order for MedBullets_df_op4_Q210
✅ Saved sentence order for MedBullets_df_op4_Q211
✅ Saved sentence order for MedBullets_df_op4_Q212
✅ Saved sentence order for MedBullets_df_op4_Q213
✅ Saved sentence order for MedBullets_df_op4_Q214
✅ Saved sentence order for MedBullets_df_op4_Q215
✅ Saved sentence order for MedBullets_df_op4_Q216
✅ Saved sentence order for MedBullets_df_op4_Q217
✅ Saved sentence order for MedBullets_df_op4_Q219
✅ Saved sentence order for MedBullets_df_op4_Q220
✅ Saved sentence order for MedBullets_df_op4_Q221
✅ Saved sentence order for MedBullets_df_op4_Q222
✅ Saved sentence order for MedBullets_df_op4_Q223


In [21]:
import os
import pandas as pd

# Define paths
folder_attr = "Attribution Scores"
folder_original = "Attribution Scores 1.5B Original"

# Loop over all files in the original folder
for filename in os.listdir(folder_original):
    if not filename.endswith(".csv"):
        continue

    original_path = os.path.join(folder_original, filename)
    attr_path = os.path.join(folder_attr, filename)

    if os.path.exists(attr_path):
        try:
            # Load both DataFrames
            df_original = pd.read_csv(original_path)
            df_attr = pd.read_csv(attr_path)

            # Merge: match where Source in attribution == Sentence in original
            merged_df = pd.merge(df_original, df_attr, left_on="Sentence", right_on="Source", how="left")

            # Save back to the original folder
            merged_df.to_csv(original_path, index=False)
            print(f"✅ Appended attribution to: {filename}")

        except Exception as e:
            print(f"❌ Error processing {filename}: {e}")
    else:
        print(f"⚠️ Skipped {filename} — not found in Attribution Scores folder")


✅ Appended attribution to: MedBullets_df_op4_Q60.csv
✅ Appended attribution to: MedBullets_df_op4_Q207.csv
✅ Appended attribution to: MedBullets_df_op4_Q110.csv
✅ Appended attribution to: MedBullets_df_op4_Q200.csv
✅ Appended attribution to: MedBullets_df_op4_Q117.csv
✅ Appended attribution to: MedBullets_df_op4_Q67.csv
✅ Appended attribution to: MedBullets_df_op4_Q119.csv
✅ Appended attribution to: MedBullets_df_op4_Q15.csv
✅ Appended attribution to: MedBullets_df_op4_Q272.csv
✅ Appended attribution to: MedBullets_df_op4_Q165.csv
✅ Appended attribution to: MedBullets_df_op4_Q69.csv
✅ Appended attribution to: MedBullets_df_op4_Q275.csv
✅ Appended attribution to: MedBullets_df_op4_Q162.csv
✅ Appended attribution to: MedBullets_df_op4_Q209.csv
✅ Appended attribution to: MedBullets_df_op4_Q12.csv
✅ Appended attribution to: MedBullets_df_op4_Q291.csv
✅ Appended attribution to: MedBullets_df_op4_Q157.csv
✅ Appended attribution to: MedBullets_df_op4_Q186.csv
✅ Appended attribution to: MedBul

✅ Appended attribution to: MedBullets_df_op4_Q4.csv
✅ Appended attribution to: MedBullets_df_op4_Q176.csv
✅ Appended attribution to: MedBullets_df_op4_Q261.csv
✅ Appended attribution to: MedBullets_df_op4_Q228.csv
✅ Appended attribution to: MedBullets_df_op4_Q33.csv
✅ Appended attribution to: MedBullets_df_op4_Q254.csv
✅ Appended attribution to: MedBullets_df_op4_Q192.csv
✅ Appended attribution to: MedBullets_df_op4_Q143.csv
✅ Appended attribution to: MedBullets_df_op4_Q285.csv
✅ Appended attribution to: MedBullets_df_op4_Q253.csv
✅ Appended attribution to: MedBullets_df_op4_Q99.csv
✅ Appended attribution to: MedBullets_df_op4_Q195.csv
✅ Appended attribution to: MedBullets_df_op4_Q144.csv
✅ Appended attribution to: MedBullets_df_op4_Q48.csv
✅ Appended attribution to: MedBullets_df_op4_Q282.csv
✅ Appended attribution to: MedBullets_df_op4_Q138.csv
✅ Appended attribution to: MedBullets_df_op4_Q34.csv
✅ Appended attribution to: MedBullets_df_op4_Q305.csv
✅ Appended attribution to: MedBull

## Calculate the Score Specifically

In [ ]:
import os
import pandas as pd

# Set the input folder
input_folder = "Attribution Scores"

# List all CSV files in the folder
csv_files = [f for f in os.listdir(input_folder) if f.endswith(".csv")]

# Process each CSV file
for csv_file in csv_files:
    file_path = os.path.join(input_folder, csv_file)
    
    # Load the attribution file
    df = pd.read_csv(file_path)

    # Safety check: ensure Score column exists
    if "Score" not in df.columns or df["Score"].isnull().all():
        print(f"⚠️ Skipped {csv_file}: no valid 'Score' column.")
        continue

    # Calculate mean score for the current QA
    mean_score = df["Score"].mean()

    # Assign Relevance category
    def get_relevance(score):
        if score == 0:
            return "Irrelevant"
        elif score > mean_score:
            return "Highly Relevant"
        else:
            return "Low Relevant"

    df["Relevance"] = df["Score"].apply(get_relevance)

    # Save the updated file back (overwrite or save new)
    df.to_csv(file_path, index=False)
    print(f"✅ Updated {csv_file} with Relevance labels.")


### The `ContextCiter` class

We can directly instantiate the `ContextCiter` class with a huggingface-style `pretrained_model_name_or_path`, together with a `context`, and a `query` (passed in as strings).

In [17]:
cc = ContextCiter.from_pretrained(model_name_or_path, context, query)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Alternatively, we can pass in a `model` and a `tokenizer`, which are instantiated from the `huggingface` library:

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
model = AutoModelForCausalLM.from_pretrained(model_name_or_path)
model.to("cuda")
cc = ContextCiter(model, tokenizer, context, query)

The `response` property of the ContextCiter class contains the response generated by the model. It is lazily generated when you access it.

In [18]:
cc.response

/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is sa

"Okay, so I need to figure out what type of GPUs the authors used in their paper. Let me start by recalling the context provided. The paper is about the Transformer model, which is a type of attention-based architecture for sequence modeling and machine translation. The user provided a detailed abstract, so I can look for specific details about the hardware used there.\n\nLooking at the abstract, it mentions that the authors achieved their results using eight GPUs, each with P100 architecture. The training took only twelve hours. So, the GPUs were likely high-end computing resources. I should check if there's any mention of specific models or hardware configurations in the abstract. It doesn't explicitly state the model, but it does mention the Transformers and the training setup.\n\nI know that P100 refers to NVIDIA's Pascal architecture, which is a high-end GPU with many cores and high memory bandwidth. Eight of these would be a powerful setup for training models, especially for task

Under the hood, the `ContextCiter` class applies a chat template to the
tokenized context and query, and then uses the model to generate a response.
That response is then stored in the `response` property.

### Attributing the response to sources within the context

To attribute the entire response and present the attributions in a human-readable format, we can use the `get_attributions` method, and pass in `as_dataframe=True`, as well as `top_k` to limit the number of sources to include in the attributions.

In [6]:
results = cc.get_attributions(as_dataframe=True, top_k=5)
results

Attributed: The authors used eight P100 GPUs in their Transformer architecture for training on the WMT 2014 English-to-German translation task.</s>


  0%|                                                                                                                                 | 0/64 [00:00<?, ?it/s]/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ch.no_grad(), ch.cuda.amp.autocast():
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 64/64 [00:03<00:00, 20.68it/s]
/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:192: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  return df.style.applymap(lambda val: _color_scale(val, max_val), subset=["Score"])


,Score,Source
0,13.856,The Transformer allows for significantly more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs.
1,12.577,"Our model achieves 28.4 BLEU on the WMT 2014 English-to-German translation task, improving over the existing best results, including ensembles, by over 2 BLEU."
2,5.301,"On the WMT 2014 English-to-French translation task, our model establishes a new single-model state-of-the-art BLEU score of 41.8 after training for 3.5 days on eight GPUs, a small fraction of the training costs of the best models from the literature."
3,1.910,"We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely."
4,1.616,"In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output."


`results` is a pandas styler object; to access the underlying dataframe:

In [7]:
results.data

,Score,Source
0,13.856492,The Transformer allows for significantly more ...
1,12.577197,Our model achieves 28.4 BLEU on the WMT 2014 E...
2,5.300751,On the WMT 2014 English-to-French translation ...
3,1.910178,"We propose a new simple network architecture, ..."
4,1.615602,"In this work we propose the Transformer, a mod..."


Alternatively, `.get_attributions()` can return the attribution scores as a `numpy` array, where the `i`th entry corresponds to the attribution score for the `i`th source in the context.

In [8]:
raw_results = cc.get_attributions()
raw_results

Attributed: The authors used eight P100 GPUs in their Transformer architecture for training on the WMT 2014 English-to-German translation task.</s>


array([-0.        , -0.04189325,  1.0423193 , -0.        ,  1.9101778 ,
       -0.527458  , 12.5771966 ,  5.3007509 ,  0.        ,  0.        ,
       -0.        , -0.51422529, -0.26642893,  0.43490208, -0.3266307 ,
        0.        ,  1.05068886, -0.        ,  1.61560223, 13.85649181])

We can then match these attributions to the sources using the `sources` property:

In [9]:
list(zip(cc.sources, raw_results))[:5]

[('Attention Is All You Need', np.float64(-0.0)),
 ('Abstract', np.float64(-0.04189324982279096)),
 ('The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder.',
  np.float64(1.0423192977905273)),
 ('The best performing models also connect the encoder and decoder through an attention mechanism.',
  np.float64(-0.0)),
 ('We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely.',
  np.float64(1.9101777961484763))]

### Attributing parts of the response

`.get_attributions()` optionally takes in `start_idx` and `end_idx` to
attribute only a part of the response.

To make it easier to attribute parts of the response, the `ContextCiter` class
has a utility property `response_with_indices` that contains the response annotated with
the index of each word within the response. You can access this with
`cc.response_with_indices`.

In [10]:
print(cc.response_with_indices)

[0]The [4]authors [12]used [17]eight [23]P100 [28]GPUs [33]in [36]their [42]Transformer [54]architecture [67]for [71]training [80]on [83]the [87]WMT [91]2014 [96]English[103]-[104]to[106]-[107]German [114]translation [126]task.</s[134]>


For example, we can attribute a part of the response like so:

In [12]:
start, end = 17, 32
cc.get_attributions(start_idx=start, end_idx=end, as_dataframe=True, top_k=5)

Attributed: eight P100 GPUs


/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:192: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  return df.style.applymap(lambda val: _color_scale(val, max_val), subset=["Score"])


,Score,Source
0,13.384,The Transformer allows for significantly more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs.
1,2.426,"On the WMT 2014 English-to-French translation task, our model establishes a new single-model state-of-the-art BLEU score of 41.8 after training for 3.5 days on eight GPUs, a small fraction of the training costs of the best models from the literature."
2,0.316,"Aligning the positions to steps in computation time, they generate a sequence of hidden states ht, as a function of the previous hidden state ht-1 and the input for position t. This inherently sequential nature precludes parallelization within training examples, which becomes critical at longer sequence lengths, as memory constraints limit batching across examples."
3,0.234,"The fundamental constraint of sequential computation, however, remains."
4,0.119,"In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output."


In [13]:
start, end = 83, 129
cc.get_attributions(start_idx=start, end_idx=end, as_dataframe=True, top_k=5)

Attributed: the WMT 2014 English-to-German translation task


/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:192: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  return df.style.applymap(lambda val: _color_scale(val, max_val), subset=["Score"])


,Score,Source
0,12.398,"Our model achieves 28.4 BLEU on the WMT 2014 English-to-German translation task, improving over the existing best results, including ensembles, by over 2 BLEU."
1,0.646,"On the WMT 2014 English-to-French translation task, our model establishes a new single-model state-of-the-art BLEU score of 41.8 after training for 3.5 days on eight GPUs, a small fraction of the training costs of the best models from the literature."
2,0.224,1 Introduction
3,0.042,The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder.
4,0.008,"Attention mechanisms have become an integral part of compelling sequence modeling and transduction models in various tasks, allowing modeling of dependencies without regard to their distance in the input or output sequences [2, 19]."


## Rerun with reducing irrelevant sentences

In [ ]:
import os
import re
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

# Input/output folders
cleaned_folder = "Attribution Scores 1.5 Cleaned"
output_dir = "Attribution Scores 1.5 Final"
os.makedirs(output_dir, exist_ok=True)

# Loop over df_op4 (must already be defined in the notebook)
for idx, row in df_op4.iloc[1:298].iterrows():
    opa, opb, opc, opd = row["opa"], row["opb"], row["opc"], row["opd"]
    
    # Use matching QA file
    qa_id = f"MedBullets_df_op4_Q{idx + 1}"
    input_path = os.path.join(cleaned_folder, f"{qa_id}.csv")

    if not os.path.exists(input_path):
        print(f"⚠️ File not found: {input_path}")
        continue

    try:
        df_cleaned = pd.read_csv(input_path)
        if "Source" not in df_cleaned.columns:
            print(f"⚠️ Missing 'Source' column in {qa_id}")
            continue

        # Concatenate all source sentences to form the context
        context_text = " ".join(df_cleaned["Source"].dropna().astype(str))

        # Get the query part from df_op4
        full_question = row["question"]
        _, query_text = split_question(full_question)

        # Compose query prompt with options
        query_full = (
            f"{query_text}\n"
            f"A. {opa}\nB. {opb}\nC. {opc}\nD. {opd}\n\n"
            "Read the question and state your answer. "
            "State your answer, starting with 'Answer:', ending with two line breaks.\n\n"
        )

        # Generate attribution results and response
        cc = ContextCiter(
            model,
            tokenizer,
            context=context_text,
            query=query_full,
            generate_kwargs={"max_new_tokens": 2048, "do_sample": False}
        )

        # Parse extracted answer
        raw_response = cc.response.strip()
        match = re.search(r"Answer:\s*([A-D])", raw_response)
        extracted_answer = match.group(1).strip() if match else None

        # Get attribution scores
        result = cc.get_attributions(as_dataframe=True)
        if hasattr(result, "data"):  # Handle if returned as Styler
            result = result.data

        if isinstance(result, pd.DataFrame):
            result["row_index"] = idx
            result["QA_ID"] = qa_id
            result["Extracted_Answer"] = extracted_answer

            filename = f"{qa_id}.csv".replace(" ", "_")
            result.to_csv(os.path.join(output_dir, filename), index=False)
            print(f"✅ Saved attribution and answer for {qa_id}")
        else:
            print(f"⚠️ Unexpected result type for {qa_id}: {type(result)}")

    except Exception as e:
        print(f"❌ Error processing {qa_id}: {e}")


/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:410: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:415: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Attributed: Okay, so I'm trying to figure out the underlying cause of this patient's presentation. Let me start by going through the information given. The patient is a 9-year-old girl who presented with a fever, changes in behavior, and last night her daughter felt ill. Her mother says her condition has been worsening. She's up to date on vaccinations and hasn't had infections yet. She's been treated for an E. coli infection and a urinary tract infection.

First, I'll look at the symptoms. She has a high fever at 99.5°F, which is pretty high, so that's a red flag. Her blood pressure is 60/35 mmHg, which is a bit low for her age, but maybe she's on some sort of medication. Her pulse is 190/min, which is quite high—over 120, so that's a lot. Respiration is 33/min, which is also high. Oxygen saturation is 98%, which is pretty good, so she's probably getting enough oxygen.

She's been treated for E. coli and UTI, which are common infections. But the question is about the underlying cause,

  0%|                                                                                                  | 0/64 [00:00<?, ?it/s]/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ch.no_grad(), ch.cuda.amp.autocast():
  0%|                                                                                                  | 0/64 [00:01<?, ?it/s]
/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:410: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:415: UserWarning: `do_sample` is 

❌ Error processing MedBullets_df_op4_Q2: CUDA out of memory. Tried to allocate 1.31 GiB. GPU 0 has a total capacity of 79.14 GiB of which 1.20 GiB is free. Process 890491 has 70.98 GiB memory in use. Including non-PyTorch memory, this process has 6.94 GiB memory in use. Of the allocated memory 5.94 GiB is allocated by PyTorch, and 516.52 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
Attributed: Okay, so I'm trying to figure out which condition is most strongly associated with the patient described. Let's break down the information given.

The patient is a 1-year-old girl with increasing seizure frequency over two months. She's seen a neurologist, so it's likely a neurological issue. The physical exam findings include hypopigmented macules on the ski

  0%|                                                                                                  | 0/64 [00:00<?, ?it/s]/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ch.no_grad(), ch.cuda.amp.autocast():
100%|█████████████████████████████████████████████████████████████████████████████████████████| 64/64 [00:30<00:00,  2.12it/s]
/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:192: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  return df.style.applymap(lambda val: _color_scale(val, max_val), subset=["Score"])
/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:410: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used

...

✅ Saved attribution and answer for MedBullets_df_op4_Q91
Attributed: Okay, so I'm trying to figure out the diagnosis for this patient. Let's start by looking at the information given. The patient is a 27-year-old man who presented with pain in his right leg after landing off a ski jump. He's crying out in pain, which suggests he's not just feeling it but is actively complaining. The physical exam shows swelling over his right leg and knee, and he's crying out. He's asking for medications and is crying out, which is a strong indicator of pain.

Looking at the symptoms, the main issue is pain in the leg and knee. The radiograph shows swelling, which could be due to various conditions. The options are patellar fracture, apophysitis of the tibial tubercle, posterior dislocation of the knee, and Tibial plateau fracture.

First, I'll consider patellar fracture. That would involve the patella, the lower leg's foot, and could cause swelling in the leg and knee. However, the patient is crying o

  0%|                                                                                    | 0/64 [00:00<?, ?it/s]/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ch.no_grad(), ch.cuda.amp.autocast():
 95%|███████████████████████████████████████████████████████████████████████▍   | 61/64 [00:14<00:00,  4.12it/s]

Attributed: Okay, so I'm trying to figure out the diagnosis for this patient. Let's break down the information given. The patient was discharged a week ago for pneumonia treated with azithromycin. Now he's feeling symptoms may return. He drinks 1-2 beers per day and smokes 1 pack of cigarettes. He recently went camping and hiking in the woods. The skin lesions are thick and don't break when pressure is applied. He's currently sexually active with multiple partners and doesn't use condoms. He had an itchy reaction recently, then noticed skin lesions that broke out prompting him to the emergency department.

First, I need to consider the possible causes of skin lesions. The patient has been exposed to alcohol and smoking, which are known to trigger skin conditions. Beers and cigarettes are common triggers for skin reactions, especially when pressure is applied. The thick skin lesions that don't break when pressure is applied might indicate a reaction to alcohol or smoking.

Looking at th

  0%|                                                                                    | 0/64 [00:00<?, ?it/s]/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ch.no_grad(), ch.cuda.amp.autocast():
100%|███████████████████████████████████████████████████████████████████████████| 64/64 [00:14<00:00,  4.44it/s]
/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:192: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  return df.style.applymap(lambda val: _color_scale(val, max_val), subset=["Score"])
/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:410: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation 

✅ Saved attribution and answer for MedBullets_df_op4_Q93


In [9]:
import os
import re
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

# Input/output folders
cleaned_folder = "Attribution Scores 1.5 Cleaned"
output_dir = "Attribution Scores 1.5 Final"
os.makedirs(output_dir, exist_ok=True)

# Get the list of already processed QA_IDs from the output folder
completed_files = set(os.listdir(output_dir))
completed_ids = {filename.replace(".csv", "") for filename in completed_files}

# Loop through df_op4 as before
for idx, row in df_op4.iloc[1:298].iterrows():
    qa_id = f"MedBullets_df_op4_Q{idx + 1}"

    if qa_id in completed_ids:
        print(f"⏩ Skipping {qa_id} (already processed)")
        continue

    input_path = os.path.join(cleaned_folder, f"{qa_id}.csv")
    if not os.path.exists(input_path):
        print(f"⚠️ File not found: {input_path}")
        continue

    try:
        df_cleaned = pd.read_csv(input_path)
        if "Source" not in df_cleaned.columns:
            print(f"⚠️ Missing 'Source' column in {qa_id}")
            continue

        # Form context and query
        context_text = " ".join(df_cleaned["Source"].dropna().astype(str))
        full_question = row["question"]
        _, query_text = split_question(full_question)

        query_full = (
            f"{query_text}\n"
            f"A. {row['opa']}\nB. {row['opb']}\nC. {row['opc']}\nD. {row['opd']}\n\n"
            "Read the question and state your answer. "
            "State your answer, starting with 'Answer:', ending with two line breaks.\n\n"
        )

        # Run ContextCiter
        cc = ContextCiter(
            model,
            tokenizer,
            context=context_text,
            query=query_full,
            generate_kwargs={"max_new_tokens": 2048, "do_sample": False}
        )

        raw_response = cc.response.strip()
        match = re.search(r"Answer:\s*([A-D])", raw_response)
        extracted_answer = match.group(1).strip() if match else None

        result = cc.get_attributions(as_dataframe=True)
        if hasattr(result, "data"):
            result = result.data

        if isinstance(result, pd.DataFrame):
            result["row_index"] = idx
            result["QA_ID"] = qa_id
            result["Extracted_Answer"] = extracted_answer

            filename = f"{qa_id}.csv".replace(" ", "_")
            result.to_csv(os.path.join(output_dir, filename), index=False)
            print(f"✅ Saved attribution and answer for {qa_id}")
        else:
            print(f"⚠️ Unexpected result type for {qa_id}: {type(result)}")

    except Exception as e:
        print(f"❌ Error processing {qa_id}: {e}")

/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is sa

Attributed: Okay, so I'm trying to figure out the underlying cause of this patient's presentation. Let me start by going through the information given. The patient is a 9-year-old girl who presented with a fever, changes in behavior, and last night her daughter felt ill. Her mother says her condition has been worsening. She's up to date on vaccinations and hasn't had infections yet. She's been treated for an E. coli infection and a urinary tract infection.

First, I'll look at the symptoms. She has a high fever at 99.5°F, which is pretty high, so that's a red flag. Her blood pressure is 60/35 mmHg, which is a bit low for her age, but maybe she's on some sort of medication. Her pulse is 190/min, which is quite high—over 120, so that's a lot. Respiration is 33/min, which is also high. Oxygen saturation is 98%, which is pretty good, so she's probably getting enough oxygen.

She's been treated for E. coli and UTI, which are common infections. But the question is about the underlying cause,

  0%|                                                                                                  | 0/64 [00:00<?, ?it/s]/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ch.no_grad(), ch.cuda.amp.autocast():
100%|█████████████████████████████████████████████████████████████████████████████████████████| 64/64 [00:08<00:00,  7.34it/s]
/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:192: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  return df.style.applymap(lambda val: _color_scale(val, max_val), subset=["Score"])
/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used

✅ Saved attribution and answer for MedBullets_df_op4_Q2
⏩ Skipping MedBullets_df_op4_Q3 (already processed)
Attributed: Okay, so I'm trying to figure out the best next step for this patient. Let me start by understanding the situation. The patient is a 17-year-old boy who's been diagnosed with a high-carb diet and has gained 20 pounds since high school. He's had a bad skin issue that hasn't improved with home remedies. He's been prescribed benzoyl peroxide and topical retinoids. He also has persistent face lesions since he was 13.

The physical exam shows some lesions, and he's returned a month later with similar symptoms. The options given are A through D, and I need to choose the most appropriate next step.

First, I should consider the possible causes of the skin lesions. Since he's on retinoids, which are often used for acne, but he's also on a high-carb diet. High-carb diets can lead to weight gain and may cause acne, but it's not always the case. Another possibility is that the l

  0%|                                                                                                  | 0/64 [00:00<?, ?it/s]/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ch.no_grad(), ch.cuda.amp.autocast():
100%|█████████████████████████████████████████████████████████████████████████████████████████| 64/64 [00:07<00:00,  9.00it/s]
/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:192: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  return df.style.applymap(lambda val: _color_scale(val, max_val), subset=["Score"])
/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used

...

  0%|                                                                                    | 0/64 [00:00<?, ?it/s]/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ch.no_grad(), ch.cuda.amp.autocast():
100%|███████████████████████████████████████████████████████████████████████████| 64/64 [00:03<00:00, 19.38it/s]
/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:192: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  return df.style.applymap(lambda val: _color_scale(val, max_val), subset=["Score"])
/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation 

✅ Saved attribution and answer for MedBullets_df_op4_Q307
Attributed: Okay, so I need to figure out the potential side effect of the medication the patient received. Let me start by understanding the context. The patient is a 26-year-old woman who presented with her heart feeling "beating out of her chest." She tried various methods like vagal maneuvers but failed, so the emergency physician gave her an IV medication. Her symptoms are a low temperature, high blood pressure, a high pulse, low respirations, and a high oxygen saturation on room air. She also mentioned that the effect of the medication wears off quickly.

Now, the question is asking which of the options A to D is a potential side effect. The options are Tachycardia, Photosensitivity, Flushing, or Seizure.

First, I'll consider each option one by one.

Option A: Tachycardia. This is rapid heartbeating. The patient's symptoms include a low temperature and high blood pressure, which are related to the heart. However, the medi

  0%|                                                                                    | 0/64 [00:00<?, ?it/s]/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ch.no_grad(), ch.cuda.amp.autocast():
100%|███████████████████████████████████████████████████████████████████████████| 64/64 [00:02<00:00, 22.15it/s]


✅ Saved attribution and answer for MedBullets_df_op4_Q308


/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:192: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  return df.style.applymap(lambda val: _color_scale(val, max_val), subset=["Score"])
